# Logit modelling for the Titanic challenge
In this notebook, I'll show the usage of Logit to produce predictions for the Titanic challenge.

I'll start by importing/cleaning data. Then, I'll show the results of a naiive model as our benchmark and then compare Logit with the naiive benchmark.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#importing files to memory
def load_dataset(filename, target_col=None, dropna_target=False, **opts):
    '''
    Loads a dataset from a csv file and splits the target variable in a separate column
    target_col is a string with the name of the target column. If this argument is not passed, will just return pandas.read_csv's output
    dropna_target = True will drop all rows where the target variable is non-numeric
    **opts are arguments that will be passed to the pandas.read_csv function
    '''
    import pandas as pd

    X = pd.read_csv(filename, **opts)

    if dropna_target==True:
        X = X.dropna(axis=0, subset=[target_col])
    
    if target_col == None:
        return X

    if target_col not in X.columns:
        raise RuntimeError('target_col not found in the especified file')

    y = X[target_col]
    X = X.drop([target_col], axis=1)
    return X, y

X, y = load_dataset('../input/titanic/train.csv', 'Survived', dropna_target=True)
Xy = X.copy()
Xy['Survived'] = y.copy()
X.head(3)

In [ ]:
sns.catplot(data=Xy, x='Sex', y='Survived', hue='Pclass', kind='bar')

## A first model
Let's use a naiive approach to be our accuracy benchmark. Based on the chart above, we'll assume that all 1st and 2nd class women survived and 1st class males also survived. All the others will be assumed to have perished.

A second naiive approach in which all females survive and all males perish will be tested.

In [ ]:
#Building a naiive prediction model 
# 1st class males and 1st+2nd class females will be predicted as survived
predictions_naiive = {('male', 1) : 1,
               ('male', 2) : 0,
               ('male', 3) : 0,
               ('female', 1) : 1,
               ('female', 2) : 1,
               ('female', 3) : 0}

#women survive, males perish
predictions_gender = {'male': 0,
                      'female': 1}


# loading the test data
X_test = load_dataset('../input/titanic/test.csv')

In [ ]:
#Making the naiive predictions for the test dataset
y_test = [predictions_naiive[x] for x in zip(X_test['Sex'], X_test['Pclass'])]

output_df = pd.DataFrame(zip(X_test['PassengerId'], y_test),
                         columns=['PassengerId','Survived'])

output_df.to_csv('naiive_submission.csv', index=False)

In [ ]:
#Checking how the naiive approaches perform in the train set
y_naiive = [predictions_naiive[x] for x in zip(X['Sex'], X['Pclass'])]
y_gender = [predictions_gender[x] for x in X['Sex']]
naiive_perf = sum(y_naiive == y)/len(y)
gender_perf = sum(y_gender == y)/len(y)
print(f'Naiive model\'s accuracy: {naiive_perf:.3f} \n')
print(f'Gender model\'s accuracy: {gender_perf:.3f}.')

## Logistic regression approach
The naiive approach gave us a 0.715 score in the official test dataset, which is lower than the 0.766 given by the "all females survive" model. Which is consistent yet lower than what we've obtained in the test set. Let's try a logistic regression approach to improve the score.

The benchmark to beat is the scores obtained by the "gender-only" naiive model. 

I'll make some more serious EDA, treat features, and then run a logistic model. Let's start by running some basic analyses to get acquainted with the training data.

In [ ]:
X.describe()

In [ ]:
X.isna().sum()/len(X)

Too many missing value in the age and cabin variables. Cabin will be ignored and we'll fill age values w/ the method described in [this Kaggle notebook](https://www.kaggle.com/code/maheshnanavare/titanic-using-pipelined-xgboost-gridsearch). We'll also scale the variables to the [0,1] range.

In [ ]:
#Creating the prefix variable
#Using code from https://www.kaggle.com/code/maheshnanavare/titanic-using-pipelined-xgboost-gridsearch

prefixes = X['Name'].str.split(expand=True)[1].value_counts()
top_prefixes = prefixes[prefixes>len(X)/25].index

X['Prefix'] = X['Name'].str.split(expand=True)[1]
X_test['Prefix'] = X_test['Name'].str.split(expand=True)[1]
X.Prefix = X.Prefix.apply(lambda x: x if x in top_prefixes else 'other')
X_test.Prefix = X_test.Prefix.apply(lambda x: x if x in top_prefixes else 'other')

In [ ]:
# Using the prefix variable to input missing ages

avg_ages = X.groupby(['Prefix','Sex','Pclass'])['Age'].mean().round()

for (i,j,k) in avg_ages.index:
    value=avg_ages.loc[i,j,k]
    X.loc[(X.Prefix==i) & (X.Sex==j) & (X.Pclass==k) & (X.Age.isnull()), ['Age']] = value
    X_test.loc[(X_test.Prefix==i) & (X_test.Sex==j) & (X_test.Pclass==k) & (X_test.Age.isnull()), ['Age']] = value

In [ ]:
X.isna().sum()/len(X)

I'll now move to basic feature engineering. I'll create dummy variables and scale continuous variables to 0-1.

I'll also create a "family_size" variable that sums SibSp and Parch. As you'll see further on, I'll test a hypothesis of using family_size as a replacement for them.

In [ ]:
#Scaling continuous variables to 0-1 and creating the dummies that will feed the logistic regression model

#sex_dummy
X['is_female'] = [int(s == 'female') for s in X['Sex']]
X_test['is_female'] = [int(s == 'female') for s in X_test['Sex']]

#embarked dummies
X['embarked_C'] = [int(x == 'C') for x in X['Embarked']]
X_test['embarked_C'] = [int(x == 'C') for x in X_test['Embarked']]
X['embarked_S'] = [int(x == 'S') for x in X['Embarked']]
X_test['embarked_S'] = [int(x == 'S') for x in X_test['Embarked']]

#pclass dummies
X['pclass_1'] = [int(x == 1) for x in X['Pclass']]
X_test['pclass_1'] = [int(x == 1) for x in X_test['Pclass']]
X['pclass_2'] = [int(x == 2) for x in X['Pclass']]
X_test['pclass_2'] = [int(x == 2) for x in X_test['Pclass']]


In [ ]:
#Creating the family_size variable
X['family_size'] = X['SibSp'] + X['Parch']
X_test['family_size'] = X_test['SibSp'] + X_test['Parch']

In [ ]:
#scaling continuous variables to [0,1]
cols_to_scale = ['Age', 'Fare', 'SibSp', 'Parch', 'family_size']
X = X.apply(lambda x: x/x.max() if x.name in cols_to_scale else x, axis=0)
X_test = X_test.apply(lambda x: x/x.max() if x.name in cols_to_scale else x, axis=0)

X.head(5)

In [ ]:
#subsetting X and X_test to the variables that will go to the training set
#family size is initially kept out as it is dependent on SibSp and Parch.
cols_to_train = ['Age', 'SibSp', 'Parch', 'Fare', 'is_female', 'embarked_C', 'embarked_S', 'pclass_1', 'pclass_2']
Xs = X[cols_to_train]
Xs_test = X_test[cols_to_train]

Creating a first Logit model with all the variables we've picked

In [ ]:
#creating the model
import statsmodels.api as sm
logit_model=sm.Logit(y, Xs)
result = logit_model.fit()
print(result.summary2())

Checking the performance of this first Logit model attempt

In [ ]:
y_logit1 = round(result.predict(Xs))
y_logit1

In [ ]:
logit1_perf = sum(y_logit1 == y)/len(y)
print(f'First logit model\'s accuracy: {logit1_perf:.3f} \n')

Our first Logit model had an accuracy of 0.813 in the training set, beating the naiive benchmark that scored 0.787 \o/

However, this model includes some parameters with very high p-values. Let's get rid of those parameters and try again.

I'll also try to use family size as a predictor replacing SibSp and Parch.

In [ ]:
#Further constraining our subset
cols_to_retrain = ['Age', 'family_size', 'is_female', 'embarked_S', 'pclass_1', 'pclass_2']
Xs2 = X[cols_to_retrain]
Xs2_test = X_test[cols_to_retrain]

In [ ]:
logit_model2=sm.Logit(y, Xs2)
result2 = logit_model2.fit()
print(result2.summary2())

Checking the performance of our 2nd Logit attempt

In [ ]:
y_logit2 = round(result2.predict(Xs2))
y_logit2

In [ ]:
logit2_perf = sum(y_logit2 == y)/len(y)
print(f'First logit model\'s accuracy: {logit2_perf:.3f} \n')

Despite solving the high p-values issue, the performance deteriorated vs our first attempt. Let's try to bring back SibSp instead of using family size.

In [ ]:
cols_to_retrain = ['Age', 'SibSp', 'is_female', 'embarked_S', 'pclass_1', 'pclass_2']
Xs2 = X[cols_to_retrain]
Xs2_test = X_test[cols_to_retrain]

logit_model3=sm.Logit(y, Xs2)
result3 = logit_model3.fit()
print(result3.summary2())

In [ ]:
y_logit3 = round(result3.predict(Xs2))
logit3_perf = sum(y_logit3 == y)/len(y)
print(f'First logit model\'s accuracy: {logit3_perf:.3f} \n')

We are back to the 1st attempt's score without the p-value issue! Let's go with this result then :)

In [ ]:
y_test3 = round(result3.predict(Xs2_test),0)
output_df = pd.DataFrame(zip(X_test['PassengerId'], y_test3),
                         columns=['PassengerId','Survived'])

output_df = output_df.astype({'Survived': 'int32'})
output_df.to_csv('submission.csv', index=False, float_format='%.0f')

This model got a 0.778 score in the test dataset, which is an improvement vs the 0.766 score obtained by the naiive gender prediction. A small yet material improvement :)